# Multimodal models: text, images and more: worked solution

Worked solution for the exercise in the lesson **[Multimodal models: text, images and more](https://ramkumarmd.com/medical-ai/generative-ai-llms/multimodal-models)** (*Generative AI & LLMs* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](06-multimodal-models.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** Multimodal models process more than one kind of data, most often images and text. Vision-language models learn a shared space in which an image and a matching description lie close together, which allows zero-shot classification and draft report generation. They inherit the shortcut problem of imaging AI: they can learn hospital, scanner or label artefacts instead of disease. Evaluations of AI-drafted radiology reports find clinically significant errors in both AI and human reports, so their role so far is assistive.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Zero-shot classification in a shared space

Toy embeddings for three chest X-ray findings, with a scanner artefact visible in some images.

In [ ]:
import numpy as np

rng = np.random.default_rng(12)
dim = 32
classes = ["normal", "pneumonia", "pleural effusion"]
concepts = rng.normal(size=(3, dim))       # what each finding "looks like" in the shared space
hospital_tag = rng.normal(size=dim)        # a scanner/hospital artefact visible in images

def normalise(v):
    return v / np.linalg.norm(v, axis=-1, keepdims=True)

# Text embeddings for prompts like "a chest X-ray showing pneumonia"
text = normalise(concepts + rng.normal(0, 0.2, (3, dim)))

def make_images(n, p_tag_by_class):
    labels = rng.integers(0, 3, n)
    tagged = rng.random(n) < np.array(p_tag_by_class)[labels]
    imgs = concepts[labels] + rng.normal(0, 3.0, (n, dim)) + 3.0 * tagged[:, None] * hospital_tag
    return normalise(imgs), labels

def zero_shot_accuracy(imgs, labels):
    predictions = (imgs @ text.T).argmax(axis=1)    # most similar text prompt
    return (predictions == labels).mean()

imgs, labels = make_images(3000, [0.1, 0.1, 0.1])
print(f"Zero-shot accuracy: {zero_shot_accuracy(imgs, labels):.1%}")

# Zero-shot accuracy: 77.8%

### Shortcuts: learning the hospital instead of the disease

Continuing from the previous block: train a classifier on embeddings at one hospital, test at a new one.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Development hospital: pneumonia patients mostly imaged on the tagged portable scanner
dev_imgs, dev_labels = make_images(3000, [0.1, 0.9, 0.1])
probe = LogisticRegression(max_iter=2000).fit(dev_imgs, dev_labels)
test_same, y_same = make_images(3000, [0.1, 0.9, 0.1])
test_new, y_new = make_images(3000, [0.5, 0.5, 0.5])   # new hospital: scanner unrelated to diagnosis
print(f"Fine-tuned classifier, same hospital: {probe.score(test_same, y_same):.1%}")
print(f"Fine-tuned classifier, new hospital:  {probe.score(test_new, y_new):.1%}")
print(f"Zero-shot, new hospital:              {zero_shot_accuracy(test_new, y_new):.1%}")

# Fine-tuned classifier, same hospital: 84.7%
# Fine-tuned classifier, new hospital:  67.6%
# Zero-shot, new hospital:              72.6%

## Exercise

**Goal:** Explore zero-shot classification and shortcut learning with toy embeddings, then with an open chest X-ray dataset if available.

**Data:** Toy embeddings generated in the notebook; optional extension uses an openly licensed chest X-ray dataset (Generated in the notebook; extension dataset under its own licence)

### Step 1

Reproduce the zero-shot and shortcut results, and vary how strongly the artefact is tied to pneumonia.

In [ ]:
print(f"Zero-shot accuracy (no shortcut): {zero_shot_accuracy(imgs, labels):.1%}")
print("artefact in pneumonia   same hospital   new hospital")
for p_tag in [0.1, 0.5, 0.7, 0.9, 0.99]:
    d_imgs, d_lab = make_images(3000, [0.1, p_tag, 0.1])
    clf = LogisticRegression(max_iter=2000).fit(d_imgs, d_lab)
    s_imgs, s_lab = make_images(3000, [0.1, p_tag, 0.1])
    n_imgs, n_lab = make_images(3000, [0.5, 0.5, 0.5])
    print(f"{p_tag:>20.0%}   {clf.score(s_imgs, s_lab):>13.1%}   {clf.score(n_imgs, n_lab):>12.1%}")

The more tightly the scanner artefact is tied to pneumonia at the development hospital, the better the
classifier looks there and the worse it does at a hospital where the scanner is unrelated to diagnosis. The
same-hospital test cannot reveal the problem, because the shortcut works there too.

### Step 2

Remove the artefact direction from the embeddings (project it out) and retrain; how does new-hospital accuracy change?

In [ ]:
def make_images_tagged(n, p_tag_by_class):
    """Same as make_images, but also returns which images carry the artefact (e.g. from scanner metadata)."""
    labs = rng.integers(0, 3, n)
    tagged = rng.random(n) < np.array(p_tag_by_class)[labs]
    raw = concepts[labs] + rng.normal(0, 3.0, (n, dim)) + 3.0 * tagged[:, None] * hospital_tag
    return normalise(raw), labs, tagged

dev_x, dev_y, dev_tag = make_images_tagged(3000, [0.1, 0.9, 0.1])
new_x, new_y, _ = make_images_tagged(3000, [0.5, 0.5, 0.5])

# Estimate the artefact direction from the data: mean difference between tagged and untagged images
# within each class, so the disease signal is not removed along with it.
direction = np.mean([dev_x[(dev_y == c) & dev_tag].mean(0) - dev_x[(dev_y == c) & ~dev_tag].mean(0)
                     for c in range(3)], axis=0)
u = direction / np.linalg.norm(direction)
project_out = lambda x: x - np.outer(x @ u, u)
print(f"Cosine similarity of estimated and true artefact direction: {u @ hospital_tag / np.linalg.norm(hospital_tag):.2f}")

before = LogisticRegression(max_iter=2000).fit(dev_x, dev_y)
after = LogisticRegression(max_iter=2000).fit(project_out(dev_x), dev_y)
print(f"New hospital accuracy, original embeddings:      {before.score(new_x, new_y):.1%}")
print(f"New hospital accuracy, artefact projected out:   {after.score(project_out(new_x), new_y):.1%}")

Removing the artefact direction takes the shortcut away, so the classifier has to use the disease signal,
and it transfers better. In real images the artefact is rarely a single clean direction (it may be text
markers, positioning, image processing), so this is a diagnostic idea more than a fix: the reliable
remedies are training data where the artefact is not tied to the diagnosis, and testing at new sites.

### Step 3

Try 'prompt ensembling': average several text descriptions per class and compare zero-shot accuracy.

In [ ]:
def zero_shot_with(text_emb, x, y):
    return ((x @ text_emb.T).argmax(axis=1) == y).mean()

test_x, test_y = make_images(3000, [0.1, 0.1, 0.1])
r = np.random.default_rng(30)
singles = [zero_shot_with(normalise(concepts + r.normal(0, 0.2, (3, dim))), test_x, test_y) for _ in range(20)]
for n_prompts in [1, 3, 5, 10]:
    accs = []
    for _ in range(20):
        prompts = concepts[None] + r.normal(0, 0.2, (n_prompts, 3, dim))   # several descriptions per class
        accs.append(zero_shot_with(normalise(normalise(prompts).mean(axis=0)), test_x, test_y))
    print(f"{n_prompts:>2} prompt(s) per class: mean zero-shot accuracy {np.mean(accs):.1%} (sd {np.std(accs):.1%})")

Each text prompt is a noisy description of its class. Averaging several cancels part of that noise, so
zero-shot accuracy rises a little and, more noticeably, varies less with the exact wording chosen. With
real models, prompt ensembling uses paraphrases such as "chest X-ray with pneumonia", "radiograph showing
consolidation", "lobar pneumonia on a chest film".

### Step 4

Optional: with an open chest X-ray dataset and an open vision-language model, compare zero-shot accuracy for two sites or acquisition types.

Optional extension (outline, not run here). With an openly licensed chest X-ray dataset and an open
vision-language model:

1. **Check the licence and terms** of both the dataset and the model, and use only de-identified images.
2. **Pick two sites or acquisition types** recorded in the metadata (for example two hospitals, or frontal
   images taken with portable and fixed scanners), and a small set of labels present in both.
3. **Write two or three text prompts per label**, embed the images and prompts with the model, and classify
   each image by the most similar averaged prompt (as in step 3).
4. **Report accuracy (or AUROC per label) separately for each site/acquisition type**, with confidence
   intervals, and compare the label mix at each site: a difference in accuracy can come from case mix as well
   as from image differences.

A large gap between sites is the real-world version of the shortcut in step 1, and a reason to validate any
imaging model at each site where it will be used.

---

Lesson: [https://ramkumarmd.com/medical-ai/generative-ai-llms/multimodal-models](https://ramkumarmd.com/medical-ai/generative-ai-llms/multimodal-models). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.